# Episode 5 — `adsr~`: the Envelope

⏱ About 25 minutes · Difficulty ★★☆

**Musical element this episode controls: the shape of loudness over time (articulation)** — whether a note is "plucked" or "swelled".

**The equivalent Max patch:** `adsr~` → `*~`.

**What you will hear:**
1. one timbre and pitch with three different feels: pluck, pad, organ
2. the "click" you get without an envelope, and how it is removed
3. an 8-note bassline

**By the end you can:**
- explain why an envelope "is just another array"
- connect dots into lines with `np.interp`
- write and use `adsr()`
- say where clicks come from and how to remove them
- join several enveloped notes into a sequence

## Toolbox

Everything built in earlier episodes, copied as-is. Just run it; no need to re-read (the tricky lines carry `#` comments).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio, display

sr = 44100

def play(x, gain=0.3):
    # np.clip holds anything beyond ±1 so nothing blows up
    return Audio(np.clip(gain * np.asarray(x), -1, 1), rate=sr, normalize=False)

def scope(x, title="", seconds=0.03):
    n = int(sr * seconds)                      # only plot this many samples from the start
    plt.figure(figsize=(9, 2.5))
    plt.plot(np.arange(n) / sr, np.asarray(x)[:n])
    plt.xlabel("time (s)")
    plt.title(title)
    plt.show()

def make_table(harmonic_amps, N=2048):
    p = np.arange(N) / N                       # phase of every table position, 0 to 1
    table = np.zeros(N)                        # start from an empty table of zeros
    for k, amp in enumerate(harmonic_amps, start=1):
        # harmonic k: k cycles across the table, at level amp
        table += amp * np.sin(2 * np.pi * k * p)
    return table / np.max(np.abs(table))       # normalize: peak becomes 1

def make_frames(F=16, max_harmonics=32, N=2048):
    frames = []
    for f in range(F):
        # how many harmonics frame f gets: rising evenly from 1 to max_harmonics
        n_h = 1 + round(f * (max_harmonics - 1) / (F - 1))
        frames.append(make_table([1 / k for k in range(1, n_h + 1)], N))
    return np.array(frames)                    # stack into a 2-D array: F rows × N columns

def osc_wavetable(frames, phase, pos):
    F, N = frames.shape

    # --- which two frames are we between ---
    fpos = np.clip(pos, 0, 1) * (F - 1)        # turn 0..1 into a frame number 0..F-1, decimals allowed
    f0 = np.floor(fpos).astype(int)            # the frame below
    f1 = np.minimum(f0 + 1, F - 1)             # the frame above (never past the last one)
    ffrac = fpos - f0                          # how far above the lower frame, 0..1

    # --- which two positions inside a table (same as Episode 3) ---
    p = phase * N
    i0 = np.floor(p).astype(int) % N           # left position
    i1 = (i0 + 1) % N                          # right position, wrapping to 0 at the end
    frac = p - np.floor(p)                     # the fractional part

    # interpolate inside each frame first, then between the two frames
    a = (1 - frac) * frames[f0, i0] + frac * frames[f0, i1]
    b = (1 - frac) * frames[f1, i0] + frac * frames[f1, i1]
    return (1 - ffrac) * a + ffrac * b

frames = make_frames()

freq = 110
dur = 2.0
t = np.arange(int(sr * dur)) / sr
phase = (freq * t) % 1

osc = osc_wavetable(frames, phase, 0.6)

`osc` is a steady 2-second tone (wavetable position 0.6). Every example in this episode lays a different envelope over it.

## 1. The simplest envelope: a fade-out

An envelope is **an array as long as the sound, with values between 0 and 1**. Multiply it with the sound sample by sample (`*~`) and it decides the loudness at every moment.

The simplest one: a straight line from 1 to 0. Last episode's `np.linspace` does exactly that.

**You should hear:** a note starting at full volume and fading evenly to silence over 2 seconds.

In [ ]:
fade_out = np.linspace(1, 0, len(t))

plt.figure(figsize=(9, 2.5))
plt.plot(t, osc * fade_out, linewidth=0.3)     # sound × envelope
plt.plot(t, fade_out, color="red")             # the envelope itself
plt.xlabel("time (s)")
plt.show()

play(osc * fade_out)

The red line is the envelope; the blue is the sound after multiplying, and its outline follows the red line. That is what the word "envelope" means.

## 2. Connecting dots: `np.interp`

A single straight line is not enough. We want shapes made of "several straight segments".

`np.interp(t, times, levels)`: give it a few (time, level) points; it draws straight lines between neighbours and returns the level at every moment in `t`. Think Max's `function` object feeding `line~`.

Three points below: (0 s, 0) → (0.1 s, 1) → (1.5 s, 0). So 0.1 seconds up to the top, then 1.4 seconds down to 0.

After the last point `np.interp` **stays at the last level** (0 here).

**You should hear:** a note that appears quickly and then fades slowly, fully silent after 1.5 seconds.

In [ ]:
points_time = [0.0, 0.1, 1.5]
points_level = [0.0, 1.0, 0.0]

env = np.interp(t, points_time, points_level)

plt.figure(figsize=(9, 2.5))
plt.plot(t, env)
plt.plot(points_time, points_level, "o", color="red")     # the three points we supplied
plt.xlabel("time (s)")
plt.ylabel("level")
plt.show()

play(osc * env)

## 3. Where clicks come from

Why have an envelope at all? Beyond expression there is a very practical reason.

Below a note is **cut off abruptly** at 0.5037 seconds (followed by silence). At that instant the waveform is not at 0; it jumps from some value straight to 0.

**You should hear:** a "click" at the moment the note ends.

**You should see:** the waveform chopped off mid-way, with a vertical jump.

In [ ]:
n_cut = int(sr * 0.5037)                       # the sample at which we cut
silence = np.zeros(int(sr * 0.5))              # 0.5 seconds of silence = a run of zeros

chopped = np.concatenate([osc[:n_cut], silence])

plt.figure(figsize=(9, 2.5))
plt.plot(chopped[n_cut - 600 : n_cut + 300])   # plot only the area around the cut
plt.title("abrupt cut")
plt.show()

play(chopped)

The fix: add a very short fade at the end (10 milliseconds here). Too short to hear as a "fade-out", but long enough for the waveform to return to 0 smoothly.

**You should hear:** a note of the same length with a clean ending and no click.

This is the most basic job of a release. Attack is the same idea: it prevents a click at the start.

In [ ]:
t_cut = np.arange(n_cut) / sr
end = n_cut / sr                                           # the time the note ends, in seconds

# stays at 1, then falls to 0 over the last 10 ms
safe_env = np.interp(t_cut, [0, end - 0.01, end], [1, 1, 0])

faded = np.concatenate([osc[:n_cut] * safe_env, silence])

plt.figure(figsize=(9, 2.5))
plt.plot(faded[n_cut - 600 : n_cut + 300])
plt.title("10 ms fade")
plt.show()

play(faded)

## 4. ADSR

In Max you press and release a key. Python has no keyboard, so we simply state how long the key was held; that number is the **gate**. `gate = 1.2` means held for 1.2 seconds, then released.

Five numbers:

| | name | unit | meaning |
|---|---|---|---|
| `a` | attack | seconds | time to rise from 0 to 1 |
| `d` | decay | seconds | time to fall from 1 to the sustain level |
| `s` | sustain | **level** (0–1) | the level held while the key is down. **Not a time.** |
| `r` | release | seconds | time to fall to 0 after release |
| `gate` | | seconds | how long the key is held |

### Built in two parts

**While held:** three points (0, 0) → (a, 1) → (a+d, s). After that `np.interp` stays at `s` by itself — which is exactly the sustain.

In [ ]:
a, d, s, r, gate = 0.2, 0.3, 0.5, 0.5, 1.2

held = np.interp(t, [0, a, a + d], [0, 1, s])

plt.figure(figsize=(9, 2.5))
plt.plot(t, held)
plt.axvline(gate, linestyle="--", color="gray")
plt.title("held: what the envelope does if the key is never released")
plt.xlabel("time (s)")
plt.show()

**After release:** fall in a straight line from the level at the moment of release down to 0 over `r` seconds.

- `(t - gate) / r`: time since release, in units of `r`. 0 at release, 1 after `r` seconds.
- `1 - ...`: flipped, so 1 at release and 0 after `r` seconds.
- `np.clip(..., 0, 1)`: before release this number exceeds 1 and after `r` seconds it drops below 0; both are held within 0 to 1.
- then multiply by `level_at_release` (the level at the moment of release).

In [ ]:
level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])      # ask about one time point only: how high at release
released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)

plt.figure(figsize=(9, 2.5))
plt.plot(t, released)
plt.axvline(gate, linestyle="--", color="gray")
plt.title("released: only the part after the dashed line is used")
plt.xlabel("time (s)")
plt.show()

**Put together:** `held` before release, `released` after. `np.where` appeared in Episode 2.

In [ ]:
env = np.where(t < gate, held, released)

plt.figure(figsize=(9, 2.5))
plt.plot(t, env)
plt.axvline(gate, linestyle="--", color="gray")
plt.title("ADSR  (dashed line = key released)")
plt.xlabel("time (s)")
plt.show()

### Wrap it in a function

In [ ]:
def adsr(t, a, d, s, r, gate):
    a, d, r = max(a, 1e-4), max(d, 1e-4), max(r, 1e-4)    # times must not be 0, or we would divide by zero below
    # while held: (0,0) → (a,1) → (a+d,s), then stays at s
    held = np.interp(t, [0, a, a + d], [0, 1, s])
    # how high the envelope is at the moment of release (attack may not be finished)
    level_at_release = np.interp(gate, [0, a, a + d], [0, 1, s])
    # fall from that level to 0 over r seconds; np.clip stops it going negative
    released = level_at_release * np.clip(1 - (t - gate) / r, 0, 1)
    return np.where(t < gate, held, released)  # held before release, released after

## 5. Three articulations

**1. pluck** — tiny attack, zero sustain.

**You should hear:** a short "dunk" gone in about 0.3 seconds. The gate lasts 1.2 seconds but the sound is over long before — because sustain is 0.

In [ ]:
pluck = adsr(t, a=0.005, d=0.3, s=0.0, r=0.1, gate=1.2)
play(osc * pluck)

**2. pad** — long attack.

**You should hear:** the sound fading in over 0.8 seconds, then fading away over 0.7 seconds after the release at 1.2 seconds.

In [ ]:
pad = adsr(t, a=0.8, d=0.2, s=0.8, r=0.7, gate=1.2)
play(osc * pad)

**3. organ** — attack and release of just 5 milliseconds, sustain at 1.

**You should hear:** full volume instantly, stopping at 1.2 seconds, with no click at either end.

In [ ]:
organ = adsr(t, a=0.005, d=0.01, s=1.0, r=0.005, gate=1.2)
play(osc * organ)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(9, 5), sharex=True)
for ax, e, name in zip(axes, [pluck, pad, organ], ["pluck", "pad", "organ"]):
    ax.plot(t, osc * e, linewidth=0.3)
    ax.plot(t, e, color="red")
    ax.set_ylabel(name)
axes[-1].set_xlabel("time (s)")
plt.show()

## 6. Gate length

The same ADSR, with only `gate` changed.

**You should hear:** a short note first (held 0.15 s), then a long one (held 1.4 s). The onset feels identical; only the time "held" differs.

Note the first one: at 0.15 seconds the attack (0.05) is done and the decay (0.2) is halfway, so the level at release lies between 1 and sustain — which is why the function computes `level_at_release`.

In [ ]:
short_note = adsr(t, a=0.05, d=0.2, s=0.6, r=0.3, gate=0.15)
long_note = adsr(t, a=0.05, d=0.2, s=0.6, r=0.3, gate=1.4)

plt.figure(figsize=(9, 2.5))
plt.plot(t, short_note, label="gate 0.15")
plt.plot(t, long_note, label="gate 1.4")
plt.legend()
plt.xlabel("time (s)")
plt.show()

display(play(osc * short_note))
display(play(osc * long_note))

## 7. The shape of the curve

Our envelope uses straight segments. But the ear does not hear loudness linearly: a straight fall sounds like "holding on, then suddenly gone".

Real synths use exponential curves: falling fast at first, then slower and slower. A simple approximation: **cube** the envelope. 0 stays 0, 1 stays 1, the middle is pushed down.

**You should hear:** two plucks. The first (straight) is "stiffer"; the second (curved) has a crisper start and a more natural tail, like a real plucked string.

In [ ]:
linear = adsr(t, a=0.005, d=0.8, s=0.0, r=0.1, gate=1.2)
curved = linear ** 3

plt.figure(figsize=(9, 2.5))
plt.plot(t, linear, label="linear")
plt.plot(t, curved, label="linear ** 3")
plt.legend()
plt.xlabel("time (s)")
plt.show()

display(play(osc * linear))
display(play(osc * curved))

## 8. A bassline

Wrap "oscillator + envelope" in a function `note()`; each call produces one complete note. Then join them with `np.concatenate`, as with the melody in Episode 1.

`gate=dur * 0.7`: each note is held for 70% of its length, leaving 30% for the release.

Inside the loop:

- `notes = []`: an empty list
- `notes.append(...)`: add one item to the end of the list
- after the loop the list holds 8 arrays, which `np.concatenate` joins into one

**You should hear:** an 8-note low pluck bassline, 0.25 seconds per note, 2 seconds in total.

In [ ]:
def note(freq, dur, a, d, s, r, pos=0.6):
    t = np.arange(int(sr * dur)) / sr
    phase = (freq * t) % 1
    env = adsr(t, a, d, s, r, gate=dur * 0.7)      # hold 70%, leave 30% for the release
    return osc_wavetable(frames, phase, pos) * env ** 2

pitches = [55.00, 55.00, 82.41, 55.00, 65.41, 55.00, 98.00, 82.41]      # A1 A1 E2 A1 C2 A1 G2 E2

notes = []
for f in pitches:
    notes.append(note(f, 0.25, a=0.003, d=0.15, s=0.2, r=0.05))

bassline = np.concatenate(notes)

plt.figure(figsize=(9, 2.5))
plt.plot(np.arange(len(bassline)) / sr, bassline, linewidth=0.3)
plt.xlabel("time (s)")
plt.show()

play(bassline)

## Common mistakes

| Symptom | Cause |
|---|---|
| A click at the start or end of a note | Attack or release is 0; or the release is longer than the time left for it and gets cut off |
| Changing sustain makes no audible difference | The gate is shorter than attack + decay, so the key is released before sustain is reached |
| Sustain was filled in as a time, e.g. `2.0` | Sustain is a level (0–1), not seconds |
| No sound at all | The envelope is all zeros. Check whether `a`, `d`, `s` are swapped |

## Exercises

**1.** Make a "reversed" feel: attack 1 second, sustain 1, release 0.01, gate 1.0. **Guess first:** what does it sound like?

<details><summary>Answer</summary>

```python
play(osc * adsr(t, a=1.0, d=0.01, s=1.0, r=0.01, gate=1.0))
```
The sound swells slowly and then stops dead — like a pluck played backwards.
</details>

**2.** Make each bassline note 0.125 seconds long and play `pitches` twice. Hint: `pitches + pitches` joins the two lists.

<details><summary>Answer</summary>

```python
notes = []
for f in pitches + pitches:
    notes.append(note(f, 0.125, a=0.003, d=0.08, s=0.2, r=0.03))
play(np.concatenate(notes))
```
</details>

**3.** Use `np.interp` to make an envelope with "two peaks": 0 → 1 → 0.2 → 1 → 0, and lay it over `osc`.

<details><summary>Answer</summary>

```python
env2 = np.interp(t, [0, 0.05, 0.5, 0.55, 1.5], [0, 1, 0.2, 1, 0])
play(osc * env2)
```
</details>

## Recap

| Max | Python |
|---|---|
| `adsr~` | `adsr(t, a, d, s, r, gate)` |
| note on / note off | `gate` (seconds held) |
| `function` + `line~` | `np.interp(t, times, levels)` |
| `*~` | `osc * env` |
| `sig~ 0` (silence) | `np.zeros(n)` |

- an envelope is an array
- attack and release need at least a few milliseconds, or you get clicks
- Episode 4 said an array can be patched into any parameter. This same `adsr()` will soon control the filter and the wavetable position too.

**Next episode:** `lores~` — the filter, and the first place in Python where a sample loop is unavoidable.